# Public-data baselines

This notebook reports the dataset statistics and reproduces the move-level and player-game-level baselines in Tables 4 and 5.

In [1]:
%pip install -q --disable-pip-version-check -r requirements/tutorials.txt

In [2]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import pandas as pd
from datasets import load_dataset
from datasets.utils import logging as datasets_logging

datasets_logging.disable_progress_bar()
datasets_logging.set_verbosity_error()

DATASET_ID = "artemlepin/chess-fraud"
DATASET_REVISION = "bd2804f268bf07c306217929db9b8dda5803392b"
SEED = 42
SYNTH_SOURCES = [
    "stockfish_1",
    "stockfish_9",
    "stockfish_15",
    "lc0_1",
    "lc0_100",
]

## Load data

In [3]:
chess_fraud = load_dataset(
    DATASET_ID, "chess_fraud", split="full", revision=DATASET_REVISION
)
chess_fraud_synth = load_dataset(
    DATASET_ID, "chess_fraud_synth", revision=DATASET_REVISION
)

dataset_overview = pd.DataFrame(
    [
        {"configuration": "chess_fraud", "split": "full", "rows": len(chess_fraud), "columns": len(chess_fraud.column_names)},
        *[
            {"configuration": "chess_fraud_synth", "split": split, "rows": len(dataset), "columns": len(dataset.column_names)}
            for split, dataset in chess_fraud_synth.items()
        ],
    ]
)
display(dataset_overview)
display(
    chess_fraud.select(range(3))
    .select_columns(["game_id", "player_id", "half_move", "move_player", "is_used"])
    .to_pandas()
)

,configuration,split,rows,columns
0,chess_fraud,full,38510,33
1,chess_fraud_synth,train,860183,44
2,chess_fraud_synth,test,214104,44


,game_id,player_id,half_move,move_player,is_used
0,01zsFrq4_w,37,1,e2e4,False
1,01zsFrq4_b,42,2,c7c6,False
2,01zsFrq4_w,37,3,d2d4,False


## Dataset statistics

### Table 1 — ChessFraud

In [4]:
TOURNAMENT_COLUMNS = [
    "tournament_id",
    "game_id",
    "player_id",
    "half_move",
    "move_player",
    "move_stockfish_15",
    "is_used",
    "is_cheating_move",
    "is_cheating_player_game",
    "is_accused_by_opponent",
]
tournament = chess_fraud.select_columns(TOURNAMENT_COLUMNS).to_pandas()
tournament["source_game_id"] = tournament["game_id"].str.rsplit("_", n=1).str[0]

physical_games = tournament.groupby("source_game_id", sort=False)
game_lengths = physical_games["half_move"].max()
game_labels = physical_games["is_cheating_player_game"].any()
player_game_labels = tournament.groupby(["game_id", "player_id"], sort=False)["is_cheating_player_game"].first()
participants = (
    tournament[["tournament_id", "player_id"]]
    .drop_duplicates()
    .groupby("tournament_id")["player_id"]
    .size()
)

def count_with_share(count, total):
    return f"{count:,} ({count / total:.1%})"

def length_summary(lengths):
    quartiles = lengths.quantile([0.25, 0.5, 0.75])
    return f"{quartiles.loc[0.5]:.0f} [{quartiles.loc[0.25]:.0f}, {quartiles.loc[0.75]:.0f}], {lengths.max():.0f}"

table_1 = pd.Series(
    {
        "Games": f"{len(game_lengths):,}",
        "Tournament participants": f"{participants.sum():,} ({' + '.join(participants.astype(str))})",
        "Unique players across both": f"{tournament['player_id'].nunique():,}",
        "Half-moves (rows)": f"{len(tournament):,}",
        "Games with cheating": count_with_share(int(game_labels.sum()), len(game_labels)),
        "Cheating half-moves": count_with_share(int(tournament['is_cheating_move'].sum()), len(tournament)),
        "Game length (plies): median [Q1, Q3], max": length_summary(game_lengths),
        "Player-games (game-sides)": f"{len(player_game_labels):,}",
        "Cheating player-games": count_with_share(int(player_game_labels.sum()), len(player_game_labels)),
        "Fair player-games": count_with_share(int((~player_game_labels).sum()), len(player_game_labels)),
    },
    name="Value",
).rename_axis("Item").to_frame()

assert len(tournament) == 38_510
assert len(game_lengths) == 505
assert len(player_game_labels) == 1_010
display(table_1)

,Value
Item,
Games,505
Tournament participants,77 (39 + 38)
Unique players across both,49
Half-moves (rows),"38,510"
Games with cheating,315 (62.4%)
Cheating half-moves,"9,405 (24.4%)"
"Game length (plies): median [Q1, Q3], max","70 [54, 95], 219"
Player-games (game-sides),"1,010"
Cheating player-games,407 (40.3%)


### Table 3 — ChessFraud-Synth

In [5]:
SYNTH_STATS_COLUMNS = ["game_id", "player_id", "rating_bin", "half_move", "is_used"]
synth = pd.concat(
    [dataset.select_columns(SYNTH_STATS_COLUMNS).to_pandas() for dataset in chess_fraud_synth.values()],
    ignore_index=True,
)
synth_player_games = synth.groupby(["game_id", "player_id"], sort=False)
synth_game_lengths = synth_player_games["half_move"].max()
players_per_bin = synth.groupby("rating_bin")["player_id"].nunique()
games_per_player = synth.groupby("player_id")["game_id"].nunique()
eligible = int(synth["is_used"].sum())

assert players_per_bin.nunique() == 1
assert games_per_player.nunique() == 1

table_3 = pd.Series(
    {
        "Rating bins (200-point, up to 2200)": f"{synth['rating_bin'].nunique():,}",
        "Players per bin": f"{players_per_bin.iloc[0]:,}",
        "Games per player": f"{games_per_player.iloc[0]:,}",
        "Player-games": f"{len(synth_game_lengths):,}",
        "Half-moves": f"{len(synth):,}",
        "Game length (plies): median [Q1, Q3], max": length_summary(synth_game_lengths),
        "Eligible decision points": f"{eligible:,}",
        "Labeled game fragments": f"{2 * eligible:,}",
        "Fair game fragments": f"{eligible:,} (50%)",
        "Cheat game fragments": f"{eligible:,} (50%)",
    },
    name="Value",
).rename_axis("Item").to_frame()

assert len(synth) == 1_074_287
assert len(synth_game_lengths) == 12_000
assert eligible == 417_207
display(table_3)

,Value
Item,
"Rating bins (200-point, up to 2200)",6
Players per bin,200
Games per player,10
Player-games,"12,000"
Half-moves,"1,074,287"
"Game length (plies): median [Q1, Q3], max","83 [70, 104], 265"
Eligible decision points,"417,207"
Labeled game fragments,"834,414"
Fair game fragments,"417,207 (50%)"


## Baselines

Specificity denotes fair-class recall, recall denotes cheat-class recall, and macro-F1 is the unweighted mean of both class F1 scores.

In [6]:
def binary_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)
    tp = np.sum(y_true & y_pred)
    fn = np.sum(y_true & ~y_pred)
    tn = np.sum(~y_true & ~y_pred)
    fp = np.sum(~y_true & y_pred)
    fair_f1 = 2 * tn / (2 * tn + fp + fn)
    cheat_f1 = 2 * tp / (2 * tp + fp + fn)
    return {
        "specificity": tn / (tn + fp),
        "recall": tp / (tp + fn),
        "macro_f1": (fair_f1 + cheat_f1) / 2,
    }

def random_predictions(y_true):
    rng = np.random.default_rng(SEED)
    return rng.random(len(y_true)) < np.mean(y_true)

def evaluate(y_true, predictions):
    return pd.DataFrame(
        {name: binary_metrics(y_true, y_pred) for name, y_pred in predictions.items()}
    ).T.rename_axis("baseline")

### Move-level baseline (Table 4)

The synthetic test split pairs each observed move with one assisted alternative sampled from the classical-engine configurations used in the study.

In [7]:
synth_move_columns = [
    "is_used",
    "move_player",
    "move_stockfish_15",
    *[f"move_{source}" for source in SYNTH_SOURCES if source != "stockfish_15"],
]
synth_moves = (
    chess_fraud_synth["test"]
    .select_columns(synth_move_columns)
    .to_pandas()
    .query("is_used")
    .reset_index(drop=True)
)
source_index = np.random.default_rng(SEED).integers(len(SYNTH_SOURCES), size=len(synth_moves))
source_moves = synth_moves[[f"move_{source}" for source in SYNTH_SOURCES]].to_numpy()
assisted_moves = source_moves[np.arange(len(synth_moves)), source_index]
stockfish_moves = synth_moves["move_stockfish_15"].to_numpy()

synth_y = np.concatenate(
    [np.zeros(len(synth_moves), dtype=bool), np.ones(len(synth_moves), dtype=bool)]
)
synth_engine = np.concatenate(
    [
        synth_moves["move_player"].to_numpy() == stockfish_moves,
        assisted_moves == stockfish_moves,
    ]
)
tournament_moves = tournament.query("is_used").copy()
tournament_y = tournament_moves["is_cheating_move"].to_numpy(dtype=bool)

move_results = pd.concat(
    {
        "ChessFraud-Synth": evaluate(
            synth_y,
            {
                "All-Cheat": np.ones(len(synth_y), dtype=bool),
                "Random": random_predictions(synth_y),
                "Engine top-1 match": synth_engine,
            },
        ),
        "ChessFraud": evaluate(
            tournament_y,
            {
                "All-Cheat": np.ones(len(tournament_y), dtype=bool),
                "Random": random_predictions(tournament_y),
                "Engine top-1 match": tournament_moves["move_player"].to_numpy() == tournament_moves["move_stockfish_15"].to_numpy(),
                "Human accusation": tournament_moves["is_accused_by_opponent"].to_numpy(dtype=bool),
            },
        ),
    },
    names=["dataset", "baseline"],
)

assert len(synth_moves) == 83_067
assert len(tournament_moves) == 28_410
display(move_results.round(3))

specificity  recall  macro_f1
dataset          baseline                                         
ChessFraud-Synth All-Cheat                 0.000   1.000     0.333
                 Random                    0.503   0.500     0.502
                 Engine top-1 match        0.611   0.658     0.634
ChessFraud       All-Cheat                 0.000   1.000     0.223
                 Random                    0.711   0.285     0.498
                 Engine top-1 match        0.579   0.629     0.570
                 Human accusation          0.667   0.594     0.610

### Player-game baseline (Table 5)

In [8]:
tournament_moves["engine_match"] = (
    tournament_moves["move_player"] == tournament_moves["move_stockfish_15"]
)
grouped = tournament_moves.groupby(["game_id", "player_id"], sort=False)

assert grouped["is_cheating_player_game"].nunique().max() == 1
assert grouped["is_accused_by_opponent"].nunique().max() == 1

player_games = pd.DataFrame(
    {
        "label": grouped["is_cheating_player_game"].first().astype(bool),
        "engine_match_rate": grouped["engine_match"].mean(),
        "human_accusation": grouped["is_accused_by_opponent"].first().astype(bool),
    }
)
game_y = player_games["label"].to_numpy()
game_results = evaluate(
    game_y,
    {
        "All-Cheat": np.ones(len(game_y), dtype=bool),
        "Engine top-1 match": player_games["engine_match_rate"].to_numpy() > 0.5,
        "Human accusation": player_games["human_accusation"].to_numpy(),
    },
)

assert len(player_games) == 1_010
display(game_results.round(3))

,specificity,recall,macro_f1
baseline,,,
All-Cheat,0.000,1.000,0.287
Engine top-1 match,0.738,0.617,0.677
Human accusation,0.718,0.516,0.618
